In [2]:
import sqlite3
import pandas as pd

df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

# clean column names: "Tenure Months" -> "tenure_months"
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={"customerid": "customer_id"})

# fix total_charges: blanks -> 0 (these are tenure-0 customers)
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce").fillna(0)

# split the flat file into logical tables
customers = df[["customer_id", "gender", "senior_citizen", "partner", "dependents",
                "city", "state", "zip_code", "latitude", "longitude"]]

services = df[["customer_id", "phone_service", "multiple_lines", "internet_service",
               "online_security", "online_backup", "device_protection",
               "tech_support", "streaming_tv", "streaming_movies"]]

billing = df[["customer_id", "tenure_months", "contract", "paperless_billing",
              "payment_method", "monthly_charges", "total_charges"]]

status = df[["customer_id", "churn_label", "churn_value",
             "churn_score", "cltv", "churn_reason"]]

# write to SQLite
conn = sqlite3.connect("../data/churn.db")
for name, table in {"customers": customers, "services": services,
                    "billing": billing, "status": status}.items():
    table.to_sql(name, conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

        name
0  customers
1   services
2    billing
3     status


In [3]:
query = """
SELECT b.contract,
       COUNT(*) AS customers,
       ROUND(AVG(s.churn_value) * 100, 1) AS churn_rate_pct
FROM billing b
JOIN status s ON b.customer_id = s.customer_id
GROUP BY b.contract
ORDER BY churn_rate_pct DESC;
"""
pd.read_sql(query, conn)

,contract,customers,churn_rate_pct
0,Month-to-month,3875,42.7
1,One year,1473,11.3
2,Two year,1695,2.8
